# VGGT Camera Pose — Untargeted PGD Attack (paper-style Huber loss)

Third in the depth / point-map / camera trilogy. Same threat model as the other two notebooks (L∞-bounded image perturbation, shared across views, 40 PGD steps); the target this time is VGGT's **camera head** (`pose_enc` = `(T, quat, fov)` in 9 dimensions).

The VGGT paper supervises the camera head with a Huber loss against the ground‑truth pose encoding:
$$\mathcal{L}_{\text{camera}} = \sum_{i=1}^{N} \|\hat{g}_i - g_i\|_\epsilon \quad\text{(smooth-L1)}$$
Maximising this loss is exactly the right direction to break the standard pose‑accuracy metric (AUC@30 over all pairs of relative poses), which is what every CO3D camera‑pose paper reports.

Two PGD passes are run side by side — same hyperparameters, two different references:

- **Pass A — paper‑faithful, vs ground truth.** Reference = GT pose encoding from CO3Dv2 annotations (PyTorch3D → OpenCV via VGGT's evaluation branch). We push the prediction directly away from GT, which is the most direct attack on the AUC@30 metric. The FoV / focal term is dropped (the GT FoV would need a resize‑aware conversion that the metric does not even use), so the loss is
$$L_A = \mathrm{Huber}(T_{\text{adv}}, T_{\text{gt}}) + \mathrm{Huber}(\hat q_{\text{adv}}, \hat q_{\text{gt}})$$
- **Pass B — mirror of depth / point‑map, vs clean prediction.** Reference = clean prediction (detached). This is the same "no GT needed" recipe the other two attacks use; the FoV term is kept here (the comparison is between two predictions on identical inputs, so it is well‑defined):
$$L_B = \mathrm{Huber}(T_{\text{adv}}, T_{\text{clean}}) + \mathrm{Huber}(\hat q_{\text{adv}}, \hat q_{\text{clean}}) + \tfrac{1}{2}\,\mathrm{Huber}(\mathrm{fov}_{\text{adv}}, \mathrm{fov}_{\text{clean}})$$

Weight split `(1.0, 1.0, 0.5)` and quaternion convention come from `training/loss.py::compute_camera_loss` so the attack maximises exactly what training minimises (the code uses L1 there, but the paper specifies Huber, and we follow the paper).

Evaluation is **AUC@30** over all pairs of relative poses (PoseDiffusion convention; also what the VGGT evaluation branch reports). We compute it three times — clean, adv‑A, adv‑B — and use pytorch3d's `so3_relative_angle` for the rotation angle to match the literature bit‑exactly.

Dataset is one sequence of CO3Dv2 `apple` (the smallest official category subset, downloaded via the single‑sequence subset of the official download script). Outputs go to `output/camera_attack/` for an evaluation script to re‑aggregate.


In [ ]:
import json
import os
import random
import sys
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import torchvision.utils as vutils

from pytorch3d.transforms import so3_relative_angle

from vggt.models.vggt import VGGT
from vggt.utils.load_fn import load_and_preprocess_images
from vggt.utils.pose_enc import extri_intri_to_pose_encoding, pose_encoding_to_extri_intri
from vggt.utils.rotation import mat_to_quat
from vggt.utils.geometry import closed_form_inverse_se3

sys.path.insert(0, str(Path(".").resolve()))
from data_utils.co3d_loader import load_co3d_sequence

In [ ]:
def seed_everything(seed: int):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Pick determinism over throughput; PGD is short and the security probe needs
    # to be reproducible. Same choice as depth_attack.ipynb.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEED = 0
seed_everything(SEED)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

model = VGGT.from_pretrained("facebook/VGGT-1B").to(device)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

In [ ]:
# CO3Dv2 single-sequence subset of `apple` (~190 MB, downloaded out-of-band)
# Three sequences are available; the loader prefers a landscape one to avoid
# VGGT's default center-crop discarding top/bottom content.
CO3D_DIR = Path("/u/nleobandung/co3d_data")
CO3D_CATEGORY = "apple"
NUM_IMAGES = 10  # standard fewview-test count from PoseDiffusion / VGGT eval branch

seq = load_co3d_sequence(CO3D_DIR, category=CO3D_CATEGORY, num_frames=NUM_IMAGES)
print(f"sequence: {seq.category}/{seq.sequence_name}  quality={seq.viewpoint_quality_score:.3f}")
print(f"frame numbers: {seq.frame_numbers}")
print(f"image[0]: {seq.image_paths[0]}")

images = load_and_preprocess_images(seq.image_paths).to(device).float()
images = images.unsqueeze(0)  # [B=1, S, C, H, W]
B, S, C, H, W = images.shape
print("images:", images.shape, "range:", float(images.min()), float(images.max()))

In [ ]:
# Build the GT pose encoding from CO3D's PyTorch3D-NDC extrinsics. The
# focal channels are not used by Pass A (we set w_focal=0 because GT FoV
# under VGGT's resize would need a careful pixel-space conversion that the
# AUC@30 metric does not depend on anyway). We still build a *consistent*
# pose encoding by passing dummy intrinsics scaled to the resized image
# size so `extri_intri_to_pose_encoding` runs without complaint and the
# quaternion convention matches VGGT's `mat_to_quat`.
gt_extr_t = torch.from_numpy(seq.extrinsics).to(device).float().unsqueeze(0)  # [1, S, 3, 4]
# Dummy intrinsics at (H, W); only the focal values matter for FoV, and we
# zero the focal term in Pass A. Picking fy = H, fx = W gives 90 deg FoV;
# any nonzero choice yields a well-defined quaternion conversion.
gt_intr_dummy = torch.zeros(1, S, 3, 3, device=device)
gt_intr_dummy[..., 0, 0] = W  # fx
gt_intr_dummy[..., 1, 1] = H  # fy
gt_intr_dummy[..., 0, 2] = W / 2
gt_intr_dummy[..., 1, 2] = H / 2
gt_intr_dummy[..., 2, 2] = 1.0

pose_gt = extri_intri_to_pose_encoding(gt_extr_t, gt_intr_dummy, (H, W),
                                      pose_encoding_type="absT_quaR_FoV").float()
print("pose_gt shape:", tuple(pose_gt.shape))
print("pose_gt sample[T|quat|fov]:", pose_gt[0, 0].tolist())

In [ ]:
# Clean forward pass — the reference for Pass B, and a no-attack baseline
# for the AUC@30 metric.
with torch.no_grad():
    clean = model(images)

pose_clean = clean["pose_enc"].float()  # [B, S, 9]
print("pose_clean shape:", tuple(pose_clean.shape))
print("pose_clean sample[T|quat|fov]:", pose_clean[0, 0].tolist())

In [ ]:
def build_pair_index(N: int) -> tuple[torch.Tensor, torch.Tensor]:
    """All unordered pairs (i, j), i < j, of frame indices. Returns two
    (N choose 2)-long tensors. Same convention as VGGT eval / PoseDiffusion."""
    i1, i2 = torch.combinations(torch.arange(N), 2, with_replacement=False).unbind(-1)
    return i1, i2


def se3_from_extrinsic(extr: torch.Tensor) -> torch.Tensor:
    """(*, 3, 4) -> (*, 4, 4) by appending [0, 0, 0, 1]. World-to-camera."""
    *batch, _, _ = extr.shape
    bottom = torch.zeros((*batch, 1, 4), device=extr.device, dtype=extr.dtype)
    bottom[..., 0, 3] = 1.0
    return torch.cat([extr, bottom], dim=-2)


def relative_pose_errors(pred_se3: torch.Tensor, gt_se3: torch.Tensor):
    """For all pairs i<j compute the relative pose gt -> pred error.
    pred_se3, gt_se3: (N, 4, 4) world-to-camera SE(3) matrices.
    Returns (rel_R_deg, rel_t_deg), each shape (N*(N-1)/2,).
    """
    N = pred_se3.shape[0]
    i1, i2 = build_pair_index(N)
    rel_gt   = gt_se3[i1]   @ closed_form_inverse_se3(gt_se3[i2])
    rel_pred = pred_se3[i1] @ closed_form_inverse_se3(pred_se3[i2])
    # Rotation: pytorch3d's so3_relative_angle clamps internally with eps=1e-4
    # which matches PoseDiffusion. Returns radians.
    rel_R_rad = so3_relative_angle(rel_pred[:, :3, :3], rel_gt[:, :3, :3], eps=1e-4)
    rel_R_deg = rel_R_rad * 180.0 / np.pi
    # Translation: angular error between unit-normalised translation
    # directions, folded into [0, 90 deg] via |cos| (camera-pose papers
    # treat t and -t as equivalent because the metric absorbs scene scale).
    # Equivalent to PoseDiffusion's `acos(sqrt(1 - loss))` where loss = 1 - cos^2.
    eps = 1e-15
    a = rel_pred[:, :3, 3]
    b = rel_gt[:, :3, 3]
    a = a / (a.norm(dim=-1, keepdim=True) + eps)
    b = b / (b.norm(dim=-1, keepdim=True) + eps)
    cos_abs = (a * b).sum(-1).abs().clamp(0.0, 1.0)
    rel_t_deg = torch.acos(cos_abs) * 180.0 / np.pi
    # Replace any residual NaN with a large default so they fall outside @30.
    rel_t_deg = torch.nan_to_num(rel_t_deg, nan=1e6, posinf=1e6, neginf=1e6)
    return rel_R_deg, rel_t_deg


def auc_at(rel_R_deg: torch.Tensor, rel_t_deg: torch.Tensor, thresh: int = 30) -> float:
    err = torch.maximum(rel_R_deg, rel_t_deg)
    bins = torch.linspace(0, thresh, thresh + 1, device=err.device)
    # histogram of bin-counts; cumulative-density mean = AUC under the
    # accuracy-vs-threshold curve, normalised to [0, 1].
    hist = torch.histc(err.float(), bins=thresh + 1, min=0, max=thresh)
    return (hist.cumsum(0) / err.numel()).mean().item()


def pose_enc_to_se3(pose_enc: torch.Tensor, image_size_hw) -> torch.Tensor:
    """VGGT pose_enc -> (S, 4, 4) world-to-camera SE(3)."""
    extr, _ = pose_encoding_to_extri_intri(pose_enc, image_size_hw,
                                            pose_encoding_type="absT_quaR_FoV",
                                            build_intrinsics=False)
    return se3_from_extrinsic(extr.squeeze(0))


# Sanity check: clean AUC vs GT — should be high (VGGT is very strong on CO3D).
gt_se3 = se3_from_extrinsic(gt_extr_t.squeeze(0)).double()
clean_se3 = pose_enc_to_se3(pose_clean, (H, W)).double()
rel_R, rel_t = relative_pose_errors(clean_se3, gt_se3)
clean_auc30 = auc_at(rel_R, rel_t, 30)
clean_auc15 = auc_at(rel_R, rel_t, 15)
clean_auc5  = auc_at(rel_R, rel_t, 5)
print(f"CLEAN  AUC@30={clean_auc30:.4f}  AUC@15={clean_auc15:.4f}  AUC@5={clean_auc5:.4f}")
print(f"CLEAN  rel R err deg: mean={float(rel_R.mean()):.3f}  max={float(rel_R.max()):.3f}")
print(f"CLEAN  rel t err deg: mean={float(rel_t.mean()):.3f}  max={float(rel_t.max()):.3f}")

In [ ]:
# Mirror of depth_attack / point_map_attack so the three probes are directly
# comparable. The user asked for the same 4/255 "barely visible" L_inf budget.
EPS = 4.0 / 255.0
STEP = 1.0 / 255.0
STEPS = 40
RANDOM_INIT = True
SHARE_DELTA_ACROSS_VIEWS = True

In [ ]:
def project_delta(delta_tensor: torch.Tensor, images_tensor: torch.Tensor, eps: float) -> torch.Tensor:
    """Project delta onto L_inf ball ∩ box that keeps images+delta in [0,1].
    Identical to depth_attack / point_map_attack."""
    shared = delta_tensor.shape[1] == 1 and images_tensor.shape[1] > 1
    if shared:
        lo_pix = (-images_tensor).amax(dim=1, keepdim=True)
        hi_pix = (1.0 - images_tensor).amin(dim=1, keepdim=True)
    else:
        lo_pix = -images_tensor
        hi_pix = 1.0 - images_tensor
    lo = torch.maximum(lo_pix, torch.full_like(lo_pix, -eps))
    hi = torch.minimum(hi_pix, torch.full_like(hi_pix, eps))
    return torch.clamp(delta_tensor, lo, hi)

In [ ]:
def camera_huber_loss(pose_pred: torch.Tensor,
                      pose_ref: torch.Tensor,
                      beta: float = 1.0,
                      w_trans: float = 1.0,
                      w_rot: float = 1.0,
                      w_focal: float = 0.5) -> torch.Tensor:
    """Paper's $L_\mathrm{camera} = \sum \|\hat g - g\|_\epsilon$ split by the
    same translation / rotation / focal weights used in training/loss.py.
    `F.smooth_l1_loss(beta=1.0)` is the standard Huber form with the
    transition point at |x|=1.
    """
    dT  = F.smooth_l1_loss(pose_pred[..., :3], pose_ref[..., :3], beta=beta, reduction="none")
    dR  = F.smooth_l1_loss(pose_pred[..., 3:7], pose_ref[..., 3:7], beta=beta, reduction="none")
    dFL = F.smooth_l1_loss(pose_pred[..., 7:],  pose_ref[..., 7:],  beta=beta, reduction="none")
    return w_trans * dT.mean() + w_rot * dR.mean() + w_focal * dFL.mean()


def run_pgd(loss_fn, label: str):
    """Untargeted L_inf PGD on a shared delta. loss_fn(pose_adv) -> scalar to
    *maximise*. Returns adversarial images, final delta, loss history, last
    predictions."""
    delta_shape = (B, 1, C, H, W) if SHARE_DELTA_ACROSS_VIEWS else (B, S, C, H, W)
    delta = torch.zeros(delta_shape, device=device, dtype=torch.float32)
    if RANDOM_INIT:
        delta.data.uniform_(-EPS, EPS)
    delta.data.copy_(project_delta(delta.data, images, EPS))
    delta.requires_grad_(True)

    torch.cuda.empty_cache()
    history = []
    for i in range(STEPS):
        if delta.grad is not None:
            delta.grad.detach_(); delta.grad.zero_()
        x_adv = images + delta
        preds = model(x_adv)
        pose_adv = preds["pose_enc"].float()
        loss = loss_fn(pose_adv)
        (g,) = torch.autograd.grad(loss, delta)
        with torch.no_grad():
            delta.add_(STEP * g.sign())
            delta.copy_(project_delta(delta, images, EPS))
        history.append(float(loss.detach()))
        if i % 5 == 0 or i == STEPS - 1:
            with torch.no_grad():
                linf = float(delta.abs().max())
            print(f"[{label}] step {i:3d}  loss={history[-1]:.6f}  ||delta||_inf={linf:.5f}")

    adv_images_t = (images + delta).detach()
    assert float(adv_images_t.min()) >= -1e-6
    assert float(adv_images_t.max()) <= 1.0 + 1e-6
    assert float((adv_images_t - images).abs().max()) <= EPS + 1e-6

    with torch.no_grad():
        adv_preds = model(adv_images_t)
    return adv_images_t, delta.detach(), history, adv_preds

In [ ]:
# Pass A — vs GT (paper-faithful Huber). FoV term zeroed because the GT
# FoV after VGGT's resize would need a pixel-space conversion that the
# AUC@30 metric does not depend on.
def loss_A(pose_pred):
    return camera_huber_loss(pose_pred, pose_gt, beta=1.0,
                              w_trans=1.0, w_rot=1.0, w_focal=0.0)

seed_everything(SEED)  # deterministic delta init within this notebook run
advA_images, deltaA, lossA_history, predA = run_pgd(loss_A, label="A")
poseA = predA["pose_enc"].float()
A_se3 = pose_enc_to_se3(poseA, (H, W)).double()
rel_R_A, rel_t_A = relative_pose_errors(A_se3, gt_se3)
adv_auc30_A = auc_at(rel_R_A, rel_t_A, 30)
adv_auc15_A = auc_at(rel_R_A, rel_t_A, 15)
adv_auc5_A  = auc_at(rel_R_A, rel_t_A, 5)
print(f"ADV-A  AUC@30={adv_auc30_A:.4f}  AUC@15={adv_auc15_A:.4f}  AUC@5={adv_auc5_A:.4f}")
print(f"ADV-A  rel R err deg: mean={float(rel_R_A.mean()):.3f}  max={float(rel_R_A.max()):.3f}")
print(f"ADV-A  rel t err deg: mean={float(rel_t_A.mean()):.3f}  max={float(rel_t_A.max()):.3f}")

In [ ]:
# Pass B — vs clean prediction (mirror of depth/point-map attacks). FoV
# term kept at 0.5 (training default) since clean vs adv are both predictions
# on the same input preprocessing — well-defined.
def loss_B(pose_pred):
    return camera_huber_loss(pose_pred, pose_clean.detach(), beta=1.0,
                              w_trans=1.0, w_rot=1.0, w_focal=0.5)

seed_everything(SEED)
advB_images, deltaB, lossB_history, predB = run_pgd(loss_B, label="B")
poseB = predB["pose_enc"].float()
B_se3 = pose_enc_to_se3(poseB, (H, W)).double()
rel_R_B, rel_t_B = relative_pose_errors(B_se3, gt_se3)
adv_auc30_B = auc_at(rel_R_B, rel_t_B, 30)
adv_auc15_B = auc_at(rel_R_B, rel_t_B, 15)
adv_auc5_B  = auc_at(rel_R_B, rel_t_B, 5)
print(f"ADV-B  AUC@30={adv_auc30_B:.4f}  AUC@15={adv_auc15_B:.4f}  AUC@5={adv_auc5_B:.4f}")
print(f"ADV-B  rel R err deg: mean={float(rel_R_B.mean()):.3f}  max={float(rel_R_B.max()):.3f}")
print(f"ADV-B  rel t err deg: mean={float(rel_t_B.mean()):.3f}  max={float(rel_t_B.max()):.3f}")

In [ ]:
summary = {
    "sequence": f"{seq.category}/{seq.sequence_name}",
    "viewpoint_quality_score": seq.viewpoint_quality_score,
    "num_frames": NUM_IMAGES,
    "num_pairs": int(NUM_IMAGES * (NUM_IMAGES - 1) // 2),
    "image_shape": list(images.shape),
    "AUC@30": {"clean": clean_auc30, "adv_A": adv_auc30_A, "adv_B": adv_auc30_B},
    "AUC@15": {"clean": clean_auc15, "adv_A": adv_auc15_A, "adv_B": adv_auc15_B},
    "AUC@5":  {"clean": clean_auc5,  "adv_A": adv_auc5_A,  "adv_B": adv_auc5_B},
    "rel_R_err_deg_mean": {
        "clean": float(rel_R.mean()),
        "adv_A": float(rel_R_A.mean()),
        "adv_B": float(rel_R_B.mean()),
    },
    "rel_t_err_deg_mean": {
        "clean": float(rel_t.mean()),
        "adv_A": float(rel_t_A.mean()),
        "adv_B": float(rel_t_B.mean()),
    },
    "linf_delta": {
        "adv_A": float(deltaA.abs().max()),
        "adv_B": float(deltaB.abs().max()),
    },
    "l2_delta_per_pixel": {
        "adv_A": float(deltaA.pow(2).mean().sqrt()),
        "adv_B": float(deltaB.pow(2).mean().sqrt()),
    },
    "loss_final": {
        "adv_A": lossA_history[-1],
        "adv_B": lossB_history[-1],
    },
}
print(json.dumps(summary, indent=2))

In [ ]:
out_dir = Path("output/camera_attack")
out_dir.mkdir(parents=True, exist_ok=True)
img_dir = Path("images/camera_attack")
img_dir.mkdir(parents=True, exist_ok=True)


def _np(t):
    return t.detach().cpu().numpy() if torch.is_tensor(t) else np.asarray(t)

def _pose_enc_list(pred):
    return np.stack([_np(p) for p in pred["pose_enc_list"]], axis=0)


image_size_hw = (H, W)
clean_extr, clean_intr = pose_encoding_to_extri_intri(pose_clean, image_size_hw)
advA_extr,  advA_intr  = pose_encoding_to_extri_intri(poseA, image_size_hw)
advB_extr,  advB_intr  = pose_encoding_to_extri_intri(poseB, image_size_hw)

# Shared bundle: GT cameras, the image set, etc. We duplicate the gt block
# into each npz so a future evaluator can read just one file.
common_kwargs = dict(
    gt_extrinsics=_np(gt_extr_t),
    gt_pose_enc=_np(pose_gt),
    clean_images=_np(images),
    image_paths=np.array(seq.image_paths),
    frame_numbers=np.array(seq.frame_numbers),
    image_hw_annotation=np.array(seq.image_hw),
)

np.savez_compressed(out_dir / "clean.npz",
    pose_enc=_np(pose_clean),
    pose_enc_list=_pose_enc_list(clean),
    extrinsics=_np(clean_extr),
    intrinsics=_np(clean_intr),
    **common_kwargs,
)

np.savez_compressed(out_dir / "adv_A.npz",
    pose_enc=_np(poseA),
    pose_enc_list=_pose_enc_list(predA),
    extrinsics=_np(advA_extr),
    intrinsics=_np(advA_intr),
    delta=_np(deltaA),
    adv_images=_np(advA_images),
    loss_history=np.array(lossA_history, dtype=np.float32),
    **common_kwargs,
)

np.savez_compressed(out_dir / "adv_B.npz",
    pose_enc=_np(poseB),
    pose_enc_list=_pose_enc_list(predB),
    extrinsics=_np(advB_extr),
    intrinsics=_np(advB_intr),
    delta=_np(deltaB),
    adv_images=_np(advB_images),
    loss_history=np.array(lossB_history, dtype=np.float32),
    **common_kwargs,
)

for label, adv in (("A", advA_images), ("B", advB_images)):
    sub = img_dir / f"adv_{label}"
    sub.mkdir(exist_ok=True)
    for s in range(adv.shape[1]):
        vutils.save_image(adv[0, s], sub / f"s{s}.png", normalize=False)

env = {
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version() if torch.backends.cudnn.is_available() else None,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}

metadata = {
    "seed": SEED,
    "model": "facebook/VGGT-1B",
    "dataset": {"root": str(CO3D_DIR), "category": CO3D_CATEGORY, "sequence": seq.sequence_name,
                "frame_numbers": seq.frame_numbers, "viewpoint_quality_score": seq.viewpoint_quality_score,
                "num_frames": NUM_IMAGES},
    "image_shape": list(images.shape),
    "attack": {
        "type": "PGD (sign step, L_inf ball)",
        "eps": EPS, "step": STEP, "steps": STEPS,
        "random_init": RANDOM_INIT,
        "share_delta_across_views": SHARE_DELTA_ACROSS_VIEWS,
        "pass_A": {"reference": "ground_truth_cameras (PyTorch3D->OpenCV)",
                   "loss": "Huber(T) + Huber(quat) [w_focal=0]"},
        "pass_B": {"reference": "clean_prediction (detached)",
                   "loss": "Huber(T) + Huber(quat) + 0.5*Huber(FoV)"},
    },
    "summary": summary,
    "env": env,
}
with open(out_dir / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("wrote:")
for p in sorted(out_dir.iterdir()):
    print(" ", p)
print(" ", img_dir, "(PNGs)")